# Earth-like

SPEEDY T31L8 with realistic orography and a climatological surface: a relaxed slab ocean, SPEEDY's slab land, and a slab sea-ice model. This run is one command:

```bash
python -m jem.main +configuration=earth-slab
```

The terrain, the climatological forcing, the 30-day ocean relaxation and the 1-day land relaxation are all set -- with the reason for each -- in `earth-slab.yaml`, so this notebook does not repeat them. The notebook below composes the same configuration in Python and calls `jem.runners.run(cfg)` -- the entry point `python -m jem.main` itself uses -- so it can plot what the run wrote.

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_module

import jem.config  # noqa: F401  -- registers the ${jem_data:}/${jcm_data:} resolvers
from jem import plot, runners

output_dir = (Path("output") / "02-01_earth").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Run it

In [ ]:
with initialize_config_module(config_module="jem.config", version_base="1.3"):
    cfg = compose(config_name="config", overrides=[
        "+configuration=earth-slab",
        f"coupled_run.output_dir={output_dir}",
        "coupled_run.subsample=3",       # 10 records out of 30 coupled days
        "coupled_run.checkpoint_path=null",
    ])
result = runners.run(cfg)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts at (`<component>-<first step>.nc`).

In [ ]:
atm = plot.open_output(output_dir, "atm")
ocn = plot.open_output(output_dir, "ocn")
lnd = plot.open_output(output_dir, "lnd")
list(lnd.data_vars)

## Plot

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)

sst = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[0], coastlines=True,
              title="Sea surface temperature [°C]")

land_temperature = lnd["land_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(land_temperature, ax=axes[1], coastlines=True,
              title="Land surface temperature [°C]")
plt.tight_layout()